# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bitshank-2338/flyrank-ai-starter-notebook/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Provisional contract:** the daily fact grain is one client, content item and report date. Use March 2026 as an exploratory mid-panel month, verify uniqueness and date coverage with queries, and reserve June 2026 as a sealed outcome period. The eventual ranking unit is a content item at a decision date; aggregate only information available before that date. The target is later observed impression decline, not a product flag or contemporaneous outcome. A false positive costs an editor’s time; this output supports review, not an automatic change.

**Scope and limits:** use the daily performance fact, with dimensions only where necessary and joins validated for cardinality. Exclude label-derived future metrics, identifiers from published outputs, and unavailable source observations. Availability must be checked with `IS TRUE`; client histories are unbalanced and missing data are not zeros. Neither this sample nor a model proves that a refresh causes recovery.

In [ ]:
%pip -q install duckdb
import duckdb
from google.colab import userdata
con = duckdb.connect()
token = userdata.get('HF_TOKEN')
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [token])
source = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
print('March coverage')
display(con.sql(f'SELECT COUNT(*) AS rows, MIN(report_date) AS first_day, MAX(report_date) AS last_day FROM {source}').df())
print('Duplicate grain groups (should be zero)')
display(con.sql(f'SELECT COUNT(*) AS duplicate_groups FROM (SELECT report_date, client_hash_id, content_hash_id FROM {source} GROUP BY 1,2,3 HAVING COUNT(*) > 1)').df())
print('Availability')
display(con.sql(f'SELECT COUNT(*) AS rows, COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available, COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_unavailable FROM {source}').df())

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features (computed only before decision time):** trailing impressions, trailing clicks, click-through rate (clicks divided by impressions when positive), mean observed search position, and days of observed search history. Each uses past daily rows only; position and rates must respect availability and missingness.

**Label:** subsequent 30-day impressions below 80% of prior 30-day impressions, among items meeting a minimum prior exposure threshold. The future period cannot enter feature construction. **Context:** report date and pseudonymous client/content IDs for windows, joins and grouped evaluation, not predictors. **Excluded:** future impressions, outcome-period clicks, the label itself, derived decline flags, product recommendations, and query-table fields whose fixed window overlaps the outcome.

A deliberately leaky experiment would include future impressions and compare its held-out score with the past-only model, then remove it. No score is claimed until the comparison actually runs.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.